# Fisher equation 
$$
    u_t = 0.1 \Delta u + u -u^2
$$

In [1]:
import sys
import os
sys.path.append(os.path.abspath(r'...'))
import scikit_tt as scikit
import numpy as np
import scipy.io as sio
import tensor_auxiliary as aux

In [2]:
data = sio.loadmat(r'...\fisher2d_data.mat')
t = data['t'][0]
x = data['x'][0]
y = data['y'][0]
u = data['u']

dx = x[2]-x[1]
dy = y[2]-y[1]
dt = t[2]-t[1]
Nt = len(t)


In [3]:
ut = aux.dudt(u,dt)
print(ut.shape)
uxx = np.zeros_like(u)
for i in range(Nt):
    uxx[i,:,:] = aux.diff_periodic(u[i,:,:],dx,axis=0,d=2)
uyy = np.zeros_like(u)
for i in range(Nt):
    uyy[i,:,:] = aux.diff_periodic(u[i,:,:],dx,axis=1,d=2)
print(uxx.shape)
print(uyy.shape)

(501, 512, 512)
(501, 512, 512)
(501, 512, 512)


In [4]:
# compute domain
T1 = 0
T2 = 50
X1 = 128
X2 = 256
Y1 = 128
Y2 = 256
# choose sample
np.random.seed(42)
nt = 10
nxny = 300
TT = np.linspace(T1, T2, nt+1)[:-1].astype(int)
XX = np.random.randint(X1, X2, nxny)
YY = np.random.randint(Y1, Y2, nxny)
points = np.column_stack((XX, YY))

XX = np.tile(XX, nt)
YY = np.tile(YY, nt)
TT = np.tile(TT, nxny)


In [5]:
U = np.array([u[TT,XX,YY].reshape(nt*nxny),
             uxx[TT,XX,YY].reshape(nt*nxny)+
              uyy[TT,XX,YY].reshape(nt*nxny)])
V = np.array([ut[TT,XX,YY].reshape(nt*nxny)])
P = [lambda t: 1, lambda t: t ,lambda t:t**2]
print(U.shape)
print(V.shape)

(2, 3000)
(1, 3000)


In [6]:
p = len(P)

core_type_1 = np.zeros([1, p, 1, 1])
core_type_1[0, 0, 0, 0] = 1

core_type_2 = np.zeros([1, p, 1, 1])
core_type_2[0, 1, 0, 0] = 1

core_type_3 = np.zeros([1, p, 1, 1])
core_type_3[0, 2, 0, 0] = 1

core_type_4 = np.zeros([1, 1, 1, 1])
core_type_4[0, 0, 0, 0] = 1

cores = [core_type_1]
cores.append(core_type_2)
cores.append(0.1*core_type_4)
coefficient_tensor = scikit.TT(cores) # uxx+uyy

cores = [core_type_2]
cores.append(core_type_1)
cores.append(1*core_type_4)
coefficient_tensor += scikit.TT(cores) # u

cores = [core_type_3]
cores.append(core_type_1)
cores.append(-1*core_type_4)
coefficient_tensor += scikit.TT(cores) # u2


xi_exact = coefficient_tensor
xi_exact_num = xi_exact.full().flatten()

# print(xi_exact_num)

In [7]:
xi = aux.mandy_cm(U, V, P, threshold=1e-5)
xi_num1 = xi.full().flatten()
xi_formatted = [f"{x:.4f}" for x in xi_num1]
print(", ".join(xi_formatted))

-0.0000, 0.1224, -0.1369, 1.0000, -0.0912, 0.1563, -1.0005, 0.0692, -0.0144


In [8]:
iter=500
d, m = U.shape 
p = len(P) 
n = p ** d  
#b0 = xi_num1
b0=range(n)
e=1e-5
for i in range(iter):
    psi=aux.build_psi(U,P,lam=1.5e-2,beta=b0,eps=1e-5)
    xi=aux.coefficient_solving(U,psi,P,V,1e-20)
    b1=xi.full().flatten()
    if abs(b1-b0).all()<e:
        break
    b0=b1
print(i)
xi_num2 = xi.full().flatten()
xi_formatted = [f"{x:.4f}" for x in xi_num2]
print(", ".join(xi_formatted))

243
0.0007, 0.1001, 0.0000, 0.9980, 0.0000, 0.0000, -0.9986, 0.0000, 0.0000


In [9]:
rel_errors = np.linalg.norm(xi_num1 - xi_exact_num) / np.linalg.norm(xi_exact_num)
print("OLS",rel_errors)
rel_errors = np.linalg.norm(xi_num2 - xi_exact_num) / np.linalg.norm(xi_exact_num)
print("IRLS",rel_errors)

OLS 0.16836180125388825
IRLS 0.0017744476863706703


In [1]:
candidates = []
for u in ['', 'u', 'u2']:
    for ux in ['', 'uxx+uyy', '(uxx+uyy)2']:
        candidates.append(u + ux)
candidates[0] = '1'
# candidates

['1',
 'uxx+uyy',
 '(uxx+uyy)2',
 'u',
 'uuxx+uyy',
 'u(uxx+uyy)2',
 'u2',
 'u2uxx+uyy',
 'u2(uxx+uyy)2']

In [13]:
idx = [i for i,val in enumerate(xi_exact_num) if val != 0]
res = [f"{xi_num2[i]}{candidates[i]}" for i in idx]
print(res)

['0.10006988193494686uxx+uyy', '0.9980138204037117u', '-0.9986253459641178u2']
